# GeoJSON Merge Analysis

This notebook combines v1-v4 geojson files, removes null geometries, and deduplicates by keeping versions with more related artefacts.

In [1]:
import json
import os
from pathlib import Path
import pandas as pd

In [3]:
# Load all versions
files = ['v1.geojson', 'v2.geojson', 'v3.geojson', 'v4.geojson']
all_features = []

for file in files:
    with open(file, 'r') as f:
        data = json.load(f)
        all_features.extend(data['features'])
        print(f"Loaded {len(data['features'])} features from {file}")

print(f"\nTotal features loaded: {len(all_features)}")

Loaded 16 features from v1.geojson
Loaded 15 features from v2.geojson
Loaded 17 features from v3.geojson
Loaded 17 features from v4.geojson

Total features loaded: 65


In [4]:
# Analyse null geometries
null_geom_count = sum(1 for f in all_features if f['geometry'] is None)
valid_features = [f for f in all_features if f['geometry'] is not None]

print(f"Features with null geometry: {null_geom_count}")
print(f"Features with valid geometry: {len(valid_features)}")

# Show null geometry features
print("Null geometry features:")
for f in all_features:
    if f['geometry'] is None:
        print(f"  - {f['properties'].get('name', 'Unknown')}")

Features with null geometry: 9
Features with valid geometry: 56
Null geometry features:
  - Lady's Fan
  - Lea Gull Skerry
  - Lord Antrim's Parlour
  - Lord Antrim's Parlour
  - Lady's Fan
  - Lea Gull Skerry
  - Lord Antrim's Parlour
  - Lady's Fan
  - Lord Antrim's Parlour


In [5]:
# Deduplication logic
feature_map = {}

for feature in valid_features:
    props = feature['properties']
    name = props.get('name', 'Unknown')
    
    # Create a simple geometry key (stringify coordinates)
    geom_key = json.dumps(feature['geometry'], sort_keys=True)
    
    key = (name, geom_key)
    
    # Count artefacts
    related = props.get('related_artefacts', [])
    if isinstance(related, str):
        artefact_count = 1
    elif isinstance(related, list):
        artefact_count = len(related)
    else:
        artefact_count = 0
    
    # Keep feature with more artefacts
    if key not in feature_map or artefact_count > feature_map[key]['artefact_count']:
        feature_map[key] = {
            'feature': feature,
            'artefact_count': artefact_count
        }

print(f"Deduplicated to {len(feature_map)} unique features")

Deduplicated to 15 unique features


In [6]:
# Extract unique features and create summary
unique_features = [item['feature'] for item in feature_map.values()]

# Create summary dataframe
summary_data = []
for feature in sorted(unique_features, key=lambda f: f['properties'].get('name', 'Unknown')):
    name = feature['properties'].get('name', 'Unknown')
    geom_type = feature['geometry']['type']
    artefacts = feature['properties'].get('related_artefacts', [])
    
    if isinstance(artefacts, str):
        artefact_count = 1
    elif isinstance(artefacts, list):
        artefact_count = len(artefacts)
    else:
        artefact_count = 0
    
    summary_data.append({
        'Name': name,
        'Geometry Type': geom_type,
        'Artefacts': artefact_count
    })

summary_df = pd.DataFrame(summary_data)
summary_df

,Name,Geometry Type,Artefacts
0,Amphitheatre,Polygon,2
1,Chimney Stacks,Polygon,7
2,Giant's Gate,Point,1
3,Giant's Loom,Point,2
4,Giant's Well,Point,1
5,Grand Causeway,Polygon,8
6,Hamilton's Seat,Point,3
7,Onion weathering,Point,1
8,Portnaboe,Polygon,1
9,Portnaboe Trap Dyke,Polygon,2


In [7]:
# Create v5.geojson
v5 = {
    'type': 'FeatureCollection',
    'features': unique_features
}

with open('v5.geojson', 'w') as f:
    json.dump(v5, f, indent=2)
